# 02 — Portal Traversal

A **portal** governs data flow between two holons. Portals are
first-class RDF entities stored in boundary graphs, discoverable via
SPARQL, with their CONSTRUCT queries attached as literals.

This notebook covers:

1. Registering a portal with a SPARQL CONSTRUCT query
2. Discovering portals via `find_portals_from`/`find_portals_to`
3. Traversing a portal and injecting the result into a target interior
4. Governed traversal with membrane validation and PROV-O provenance
5. Multi-hop path finding


## Setup

Two holons: a sensor source and a fusion target.


In [ ]:
from holonic import HolonicDataset

ds = HolonicDataset()

ds.add_holon("urn:holon:radar", "Radar Sensor")
ds.add_interior("urn:holon:radar", '''
    @prefix src: <urn:radar:> .
    <urn:track:001> a src:RadarContact ;
        src:lat 34.05 ;
        src:lon -118.25 ;
        src:confidence 0.92 .
    <urn:track:002> a src:RadarContact ;
        src:lat 34.07 ;
        src:lon -118.28 ;
        src:confidence 0.75 .
''')

ds.add_holon("urn:holon:fusion", "Fusion Target")
ds.add_boundary("urn:holon:fusion", '''
    @prefix tgt: <urn:fusion:> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .

    <urn:shapes:TrackShape> a sh:NodeShape ;
        sh:targetClass tgt:Track ;
        sh:property [
            sh:path tgt:position ;
            sh:minCount 1 ;
            sh:severity sh:Violation
        ] .
''')


## Register a transform portal

The portal's CONSTRUCT query is stored in the boundary graph as a
literal. It reshapes source-vocab data into target-vocab data.


In [ ]:
construct = '''
    PREFIX src: <urn:radar:>
    PREFIX tgt: <urn:fusion:>

    CONSTRUCT {
        ?t a tgt:Track ;
            tgt:position ?pos ;
            tgt:quality ?conf .
    }
    WHERE {
        ?t a src:RadarContact ;
            src:lat ?lat ;
            src:lon ?lon ;
            src:confidence ?conf .
        BIND(CONCAT(STR(?lat), ",", STR(?lon)) AS ?pos)
    }
'''

ds.add_portal(
    "urn:portal:radar-to-fusion",
    source_iri="urn:holon:radar",
    target_iri="urn:holon:fusion",
    construct_query=construct,
    label="Radar → Fusion",
)


## Discover portals via SPARQL

Portal discovery is always SPARQL-driven — not Python iteration over
cached lists.


In [ ]:
from_radar = ds.find_portals_from("urn:holon:radar")
for p in from_radar:
    print(f"{p.iri}  →  {p.target_iri}")


## Traverse and inject

`traverse_portal()` runs the CONSTRUCT against the dataset and
optionally appends the result into a target graph. Below we inject
into the fusion holon's interior.


In [ ]:
target_interior = "urn:holon:fusion/interior"
projected = ds.traverse_portal(
    "urn:portal:radar-to-fusion",
    inject_into=target_interior,
)
print(f"projected triples: {len(projected)}")


## Governed traversal with validation + provenance

`traverse()` composes discovery, traversal, membrane validation, and
`prov:Activity` recording as one governed operation. If validation
fails, data is NOT injected.


In [ ]:
# Clear the fusion interior first so we can rerun cleanly
ds.backend.delete_graph("urn:holon:fusion/interior")

projected, membrane = ds.traverse(
    "urn:holon:radar",
    "urn:holon:fusion",
    validate=True,
    agent_iri="urn:agent:fusion-pipeline",
)
print(f"membrane health: {membrane.health}")
print(f"violations:      {len(membrane.violations)}")


## Inspect the provenance trail

Every governed traversal writes a `prov:Activity` into the target
holon's context graph. The activity links back to the source and
target holons and records the agent and timestamp.


In [ ]:
rows = ds.query('''
    PREFIX prov: <http://www.w3.org/ns/prov#>
    SELECT ?activity ?start ?agent
    WHERE {
        GRAPH ?g {
            ?activity a prov:Activity ;
                prov:used <urn:holon:radar> ;
                prov:generated <urn:holon:fusion> ;
                prov:startedAtTime ?start ;
                prov:wasAssociatedWith ?agent .
        }
    }
    ORDER BY DESC(?start)
''')
rows


## Multi-hop path finding

Given an intermediate holon, the library finds a path from any source
to any reachable target via a single SPARQL query.


In [ ]:
ds.add_holon("urn:holon:track-db", "Track Database")
ds.add_portal(
    "urn:portal:fusion-to-db",
    source_iri="urn:holon:fusion",
    target_iri="urn:holon:track-db",
    construct_query="CONSTRUCT { ?s ?p ?o } WHERE { ?s ?p ?o }",
    label="Fusion → DB",
)

path = ds.find_path("urn:holon:radar", "urn:holon:track-db")
print([p.iri for p in path])


## Where to go next

- `03_cco_to_schemaorg` — cross-vocabulary portals at scale
- `08_scope_resolution` — find holons by predicate across the holarchy
- `09_projection_plugins` — projection pipelines as RDF-declared specs
